# Minggu 14 — Praktik: Mekanisme Sumber Gempa

**Seismologi PAGF262413** · Program Studi Sarjana Geofisika FMIPA UGM

Dua bagian: **mengukur polaritas sendiri** dari rekaman nyata Yogyakarta 2006, lalu **membaca mekanisme** dua belas gempa besar Indonesia dari tensor momennya.

**AI boleh dipakai sebebasnya.** Yang dinilai: ketepatan pembacaan polaritas, penalaran tentang mengapa data ini tidak cukup, dan tafsiran tektonik kalian.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from obspy import read
from obspy.imaging.beachball import beach
plt.rcParams['axes.grid']=True; plt.rcParams['grid.alpha']=.25

mek = pd.read_csv('data/W14_mekanisme_indonesia.csv')
st  = read('data/TF14_contoh.mseed')          # rekaman Minggu 11 dipakai ulang
ref = pd.read_csv('data/TF14_contoh_pick_rujukan.csv')
print(mek[['waktu','M','lat','lon','depth','strike','dip','rake','tempat']].to_string(index=False))

## Bagian 1 — Membaca gerakan pertama P

Tapis, cari onset, baca tanda setengah gelombang pertama. **Naik (+) = kompresi, turun (−) = dilatasi.**

In [ ]:
EVENT = 1
NAMA, NIM = 'tulis nama kalian', 'isi NIM kalian'

sel = st.select(location=f"{EVENT:02d}", component='Z').copy()
inf = ref[ref.id==f"EV{EVENT:02d}"].iloc[0]
fs  = sel[0].stats.sampling_rate
sel.detrend('demean'); sel.filter('bandpass', freqmin=2, freqmax=20, corners=4, zerophase=False)

y = sel[0].data.astype(float); t = np.arange(len(y))/fs
i = int(inf.p_ref_s*fs)

plt.figure(figsize=(12,4))
plt.plot(t, y, lw=.7); plt.axvline(inf.p_ref_s, color='r', ls='--')
plt.xlim(inf.p_ref_s-1, inf.p_ref_s+1.5); plt.xlabel('Waktu (detik)')
plt.title('Perbesar onset — ke mana simpangan pertama bergerak?')

# TUGAS: tentukan polaritasnya
jendela   = y[i:i+int(0.25*fs)]
polaritas = ...            # <-- '+' atau '-'
print('amplitudo puncak pertama:', jendela[np.argmax(np.abs(jendela))])

✍️ **Jawaban 1:** Polaritas kalian, dan seberapa yakin? Apa yang membuat pembacaan ini bisa keliru?

*(tulis di sini)*

## Bagian 2 — Mengapa lima polaritas tidak cukup

Materi kuliah memakai lima stasiun dan tetap tidak menghasilkan satu solusi.
Buktikan sendiri: berapa banyak mekanisme berbeda yang cocok dengan lima titik?

In [ ]:
# lima polaritas terukur dari materi kuliah (stasiun, polaritas, azimut perkiraan)
polar = [('TF12','-', 210), ('TF14','-', 250), ('TF16','-', 170), ('TF18','-', 290), ('TF19','+', 30)]

fig, ax = plt.subplots(figsize=(6,6))
ax.add_patch(plt.Circle((0,0), 1, fill=False, lw=2))
for s_, p, az in polar:
    r = .6
    x, yy = r*np.sin(np.radians(az)), r*np.cos(np.radians(az))
    ax.plot(x, yy, 'o', ms=16, mfc='#dc2626' if p=='+' else 'white', mec='k', mew=1.5)
    ax.text(x, yy-.16, s_, ha='center', fontsize=9)
ax.set_xlim(-1.3,1.3); ax.set_ylim(-1.3,1.3); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Bisakah kalian menggambar dua bidang tegak lurus\nyang memisahkan merah dari putih? Ada berapa cara?')

# TUGAS: coba beberapa mekanisme dan lihat mana yang tidak melanggar data
for sdr in [[335,7,113], [90,45,90], [180,60,-90]]:
    print(sdr, '-> apakah cocok dengan kelima polaritas?')

✍️ **Jawaban 2:** Berapa banyak mekanisme yang kalian temukan cocok? Berapa polaritas tambahan
yang kalian perkirakan diperlukan, dan **di azimut mana** stasiun barunya paling berguna?

*(tulis di sini)*

## Bagian 3 — Membaca mekanisme gempa Indonesia

In [ ]:
def tipe_sesar(rake):
    r = (rake + 360) % 360
    if 45 <= r < 135:  return 'naik'
    if 225 <= r < 315: return 'turun'
    return 'mendatar'

mek['tipe'] = mek.rake.apply(tipe_sesar)
print(mek.tipe.value_counts().to_string())

fig, ax = plt.subplots(1, 3, figsize=(13,4.5))
for k, (_, r) in enumerate(mek.nlargest(3,'M').iterrows()):
    ax[k].add_collection(beach([r.strike, r.dip, r.rake], xy=(0,0), width=180, facecolor='#dc2626'))
    ax[k].set_xlim(-110,110); ax[k].set_ylim(-110,110); ax[k].set_aspect('equal'); ax[k].axis('off')
    ax[k].set_title(f"{r.waktu}  M{r.M}\n{r.strike:.0f}/{r.dip:.0f}/{r.rake:.0f}  ({r.tipe})", fontsize=10)

# TUGAS: hubungan kedalaman dengan tipe sesar
print(mek.groupby('tipe').depth.describe()[['count','mean','max']].round(1).to_string())

✍️ **Jawaban 3:**

1. Berapa dari 12 gempa itu sesar naik? Mengapa proporsinya setinggi itu di Indonesia?
2. Gempa 2016 di Samudra Hindia bertipe mendatar dengan *dip* 84°. Mengapa ia berbeda dari yang lain?
3. Gempa Ambon 2006 berkedalaman 397 km. Pada kedalaman itu tekanan sangat besar — bagaimana
   pensesaran biasa masih mungkin terjadi?

*(tulis di sini)*

## Bagian 4 — SEL BERGALAT ⚠️

Ada **tiga kesalahan**.

In [ ]:
# ============ SEL BERGALAT ============
# Menentukan tipe sesar dan momen dari katalog

# (a) tipe sesar ditentukan dari dip
tipe = np.where(mek.dip < 30, 'naik', 'mendatar')

# (b) bidang sesar sebenarnya selalu yang dip-nya lebih landai
bidang_sesar = np.where(mek.dip < mek.dip2, 1, 2)

# (c) magnitudo momen dari momen skalar (M0 dalam katalog bersatuan N*m)
Mw = (2/3) * np.log10(mek.m0) - 10.7

print(pd.DataFrame({'tipe':tipe, 'bidang':bidang_sesar, 'Mw':Mw.round(2), 'M_katalog':mek.M}).head())
# ======================================

✍️ **Jawaban 4:**

| # | Baris | Kesalahannya | Akibatnya |
|:--|:--|:--|:--|
| 1 | | | |
| 2 | | | |
| 3 | | | |

## Bagian 5 — Setoran

In [ ]:
setoran = dict(nim=NIM, nama=NAMA, soal=f"W14-{EVENT:02d}",
    polaritas=polaritas,
    n_naik=int(sum(1 for _,p,_ in polar if p=='+')),
    n_mekanisme_cocok=...,          # <-- dari Bagian 2
    azimut_stasiun_paling_berguna=...,
    keyakinan_polaritas=...,        # <-- 'tinggi' / 'sedang' / 'rendah', dengan alasan di sel ✍️
)
pd.DataFrame([setoran]).to_csv(f'setoran_{NIM}_W14.csv', index=False)
setoran

✍️ **Catatan pemakaian AI** (wajib diisi): apa yang kalian tanyakan, bagian mana yang berasal
dari sana, dan bagian mana yang akhirnya kalian ubah sendiri.

*(tulis di sini)*